# Prepare data for model run

#### import packages

In [1]:
import pandas as pd

In [2]:
def generate_parallel_connections(input_df):
    result_rows = []

    for _, row in input_df.iterrows():
        source_name = row['source_name']
        target_name = row['target_name']
        occurrences = row['occurrences']

        for i in range(1, occurrences + 1):
            new_target = f"{target_name}_{i}"
            result_rows.append({'source_name': source_name, 'target_name': new_target})

            # Add a source node targeting the original target node
            result_rows.append({'source_name': f"{new_target}", 'target_name': target_name})

    result_df = pd.DataFrame(result_rows)

    return result_df

def replace_nan_with_value(dataframe, column_name, replacement_value):
    """
    Replace all NaN values in a specific column of a DataFrame with a given value.

    Parameters:
        dataframe (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column in which NaN values should be replaced.
        replacement_value: The value to replace NaN with.

    Returns:
        pd.DataFrame: A new DataFrame with NaN values replaced in the specified column.
    """
    dataframe.fillna({column_name: replacement_value}, inplace=True)
    return dataframe

#### Read in data

In [4]:
#to be replaced with real input data
data = {
    'source': [(757926.5454303586, 7207322.22114858), 
               (757926.5454303586, 7207322.22114858), 
               (807807.6960599166, 7159118.912010687), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038), 
               (681661.3396489053, 6764564.505472038)],
    'target': [(807807.6960599166, 7159118.912010687), 
               (807807.6960599166, 7159118.912010687), 
               (815843.850100283, 7134574.8782972265), 
               (637008.0862654719, 6772214.151799204), 
               (682552.0068947424, 6728692.507370236), 
               (709589.8414160666, 6750795.383092231), 
               (740186.3374190701, 6760051.547989771), 
               (715598.0882926519, 6728236.025495461), 
               (715598.0882926519, 6728236.025495461), 
               (715598.0882926519, 6728236.025495461)],
    'capacity': [10, 5, 8, 12, 6, 15, 10, 8, 5, 3]
}

coordinates_df = pd.DataFrame(data)

#### create parallel edges for model

In [5]:
# Get unique entries from both columns
unique_entries = pd.unique(coordinates_df[['source', 'target']].values.ravel('K'))

# Create a dictionary to map unique entries to unique names
name_mapping = {entry: f'node_{i+1:04d}' for i, entry in enumerate(unique_entries)}

# Create a DataFrame with the pairs
unique_coordinate_node_df = pd.DataFrame({'Unique Entry': unique_entries, 'Unique Name': [name_mapping[entry] for entry in unique_entries]})

# Replace entries with unique names
coordinates_df['source_name'] = coordinates_df['source'].map(name_mapping)
coordinates_df['target_name'] = coordinates_df['target'].map(name_mapping)

# Display the result
network_nodes_df = coordinates_df[['source_name', 'target_name', 'capacity']]

# Count the occurrences of each source-target pair
pair_counts = network_nodes_df.groupby(['source_name', 'target_name']).size().reset_index(name='count')

# Filter pairs that exist more than once
duplicate_pairs = pair_counts[pair_counts['count'] > 1][['source_name', 'target_name']]
duplicate_pairs['occurrences'] = duplicate_pairs.apply(lambda row: network_nodes_df[(network_nodes_df['source_name'] == row['source_name']) & (network_nodes_df['target_name'] == row['target_name'])].shape[0], axis=1)

# Add capacity column to duplicate_pairs
network_parallels_df = generate_parallel_connections(duplicate_pairs)

# Sort by 'source_name' and move the nodes with a pattern like "node_0000_digit" to the end
network_parallels_df = (network_parallels_df.sort_values(by=['source_name', 'source_name'], key=lambda x: x.str.contains(r'node_\d{4}_\d+')))

# Identify entries with duplicates in either 'source_name' or 'target_name'
duplicates_mask = network_nodes_df.duplicated(subset=['source_name', 'target_name'], keep=False)

# Filter out rows with duplicates
single_network_edges_df = network_nodes_df[~duplicates_mask]

single_network_edges_df = single_network_edges_df[['source_name', 'target_name']]

#### final model network creation

In [9]:
#create the complete network and add parameters
network_complete_df = pd.concat([network_parallels_df, single_network_edges_df])
# Sort by 'source_name' and move the nodes with a pattern like "node_0000_digit" to the end
network_complete_df = network_complete_df.reset_index(drop=True)
network_complete_df = (network_complete_df.sort_values(by=['source_name', 'source_name'], key=lambda x: x.str.contains(r'node_\d{4}_\d+')))
network_complete_df = network_complete_df.reset_index(drop=True)
network_complete_df['capacity'] = network_nodes_df['capacity']
network_complete_df
# Replace NaN values in 'column1' with 0
network_complete_df = replace_nan_with_value(network_complete_df, 'capacity', 9999)

# Display the model_network_df DataFrame
network_complete_df

,source_name,target_name,capacity
0,node_0001,node_0002_1,10.0
1,node_0001,node_0002_2,5.0
2,node_0003,node_0009_1,8.0
3,node_0003,node_0009_2,12.0
4,node_0003,node_0009_3,6.0
5,node_0002,node_0004,15.0
6,node_0003,node_0005,10.0
7,node_0003,node_0006,8.0
8,node_0003,node_0007,5.0
9,node_0003,node_0008,3.0
